In [1]:
import torch
import string
import torch.nn.functional as F

words=open("names.txt",'r').read().splitlines()
stoi = {c: i for i, c in enumerate(string.ascii_lowercase,start=1)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}

In [2]:
import random
block_size = 5
def build_dataset(words):
    X=[]
    Y=[]
    for w in words:
        context = block_size * [0]  
        for ch in w + ".":
            idx = stoi[ch]
            X.append(context)
            Y.append(idx)
            #print("".join(itos[i] for i in context),"-->",ch)
            context = context[1:]+ [idx]
            
    X=torch.tensor(X)
    Y=torch.tensor(Y)
    return X,Y

n1 = int(len(words) * 0.8)
n2 = int(len(words) * 0.9)
random.seed(42)
random.shuffle(words)
Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xtest,Ytest = build_dataset(words[n2:])

print(len(Xtr), len(Xdev), len(Xtest))

182625 22655 22866


In [3]:
class Linear:
    def __init__(self,fan_in,fan_out,bias = True):
        self.weights = torch.randn(fan_in,fan_out)
        self.bias = torch.randn(fan_out) if bias==True else None

    def __call__(self, x):
        self.out = x @ self.weights
        if self.bias is not None:
          self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weights] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
    def __init__(self,num_features,eps=1e-05,momentum = 0.1):
        self.num_features = num_features
        self.eps = eps
        self.momentum= momentum
        #parameters
        self.gamma = torch.ones(num_features)
        self.beta = torch.zeros(num_features)
        self.running_mean = torch.zeros(num_features)
        self.running_var = torch.ones(num_features)

    def __call__(self,x):
        xmean = x.mean(0,keepdim =True)
        xvar = x.var(0,keepdim =True)
        xhat  = (x-xmean)/torch.sqrt(xvar-self.eps)
        self.out = xhat*self.gamma + self.beta
        self.running_mean = self.running_mean*(1-self.momentum) + xmean * self.momentum 
        self.running_var = self.running_var*(1-self.momentum) + xvar * self.momentum 
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []


In [4]:
vocab_size = 27
n_embd = 10
n_hidden = 100 
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn(vocab_size,n_embd,generator = g)
layers = [Linear(n_embd * block_size , n_hidden,bias=False), BatchNorm1d(n_hidden),Tanh(),
          Linear(n_hidden,n_hidden,bias=False)            ,  BatchNorm1d(n_hidden),Tanh(),
          Linear(n_hidden,n_hidden,bias=False)            ,  BatchNorm1d(n_hidden),Tanh(),
          Linear(n_hidden,n_hidden,bias=False)            ,  BatchNorm1d(n_hidden),Tanh(),
          Linear(n_hidden,n_hidden,bias=False)            ,  BatchNorm1d(n_hidden),Tanh(),
          Linear(n_hidden,vocab_size,bias=False)            ,  BatchNorm1d(vocab_size)
         ]

with torch.no_grad():
  # last layer: make less confident
  layers[-1].gamma *= 0.1

parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True
print(sum(p.nelement() for p in parameters))

49024


In [5]:
idxx = torch.randint(0,Xtr.shape[0],(32,))
emb = C[Xtr[idxx]]
emb.shape

torch.Size([32, 5, 10])

In [6]:
max_steps = 50000
batch_size = 32
lossi = []
for i in range(max_steps):
    
    idxx = torch.randint(0,Xtr.shape[0],(batch_size,))
    emb = C[Xtr[idxx]]
    x = emb.view(-1,n_embd*block_size)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x,Ytr[idxx])

    for p in parameters:
        p.grad = None
    loss.backward()

    if i<15000:
        lr = 0.1
    else:
        lr = 0.01
    for p in parameters:
        p.data += - lr * p.grad

    # track stats
    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item() ) 

      0/  50000: 3.3225
  10000/  50000: 2.5070
  20000/  50000: 2.2013
  30000/  50000: 2.2367
  40000/  50000: 2.0791


In [7]:
context = [0] * block_size
out = []
for _ in range(20):
    
    emb = C[torch.tensor(context)]
    x = emb.view(-1,n_embd*block_size)
    for layer in layers:
        x = layer(x)
    probs = F.softmax(x, dim=1)
    idx = torch.multinomial(probs, 1, generator=g).item()

    if idx == 0:
        break
    out.append(itos[idx])
    context = context[1:]+[idx]
    
print("\n".join(out))

C:\Users\26698\AppData\Local\Temp\ipykernel_28772\3445076797.py:28: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  xvar = x.var(0,keepdim =True)


RuntimeError: probability tensor contains either `inf`, `nan` or element < 0